# Quick Commerce Data Model Design

## Business Objective
Build a Quick Commerce War Room capable of monitoring:

### Customer Intelligence
- Active Customers
- Repeat Customers
- Retention Rate
- Customer Segmentation

### Operations Intelligence
- Delivery Time
- Cancellation Rate
- Hub Performance
- Peak Hour Analysis

### Revenue Intelligence
- Revenue
- Average Order Value
- Revenue by Category
- Revenue by City

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_ais = pd.read_csv("../data/raw/aisles.csv")
df_dep = pd.read_csv("../data/raw/departments.csv")
df_ord = pd.read_csv("../data/raw/orders.csv")
df_prod = pd.read_csv("../data/raw/products.csv")
df_opp = pd.read_csv("../data/raw/order_products__prior.csv")

# Product Dimension Design

Goal:
Create dim_product by enriching products with aisle and department information and assigning realistic product prices.

In [3]:
dim_product = (
    df_prod
    .merge(df_ais, on="aisle_id", how="left")
    .merge(df_dep, on="department_id", how="left")
)

In [5]:
dim_product.shape

(49688, 6)

In [6]:
dim_product.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [7]:
dim_product["department"].value_counts()

department
personal care      6563
snacks             6264
pantry             5371
beverages          4365
frozen             4007
dairy eggs         3449
household          3085
canned goods       2092
dry goods pasta    1858
produce            1684
bakery             1516
deli               1322
missing            1258
international      1139
breakfast          1115
babies             1081
alcohol            1054
pets                972
meat seafood        907
other               548
bulk                 38
Name: count, dtype: int64

## Pricing Model Design

Revenue is not available in the source dataset.

To enable revenue analytics, a department-based pricing model is used to assign realistic product prices.

In [8]:
price_ranges = {
    "produce": (20, 120),
    "dairy eggs": (30, 250),
    "beverages": (20, 300),
    "bakery": (25, 250),
    "deli": (50, 500),
    "frozen": (80, 600),
    "snacks": (10, 350),
    "pantry": (20, 500),
    "breakfast": (30, 400),
    "canned goods": (25, 350),
    "dry goods pasta": (30, 400),
    "meat seafood": (100, 1200),
    "alcohol": (200, 2500),
    "personal care": (50, 1500),
    "household": (50, 2000),
    "babies": (100, 3000),
    "pets": (100, 2500),
    "international": (50, 1000),
    "bulk": (200, 5000),
    "other": (50, 500),
    "missing": (50, 500)
}

In [9]:
import numpy as np

np.random.seed(42)

dim_product["unit_price"] = dim_product["department"].apply(
    lambda x: np.random.randint(
        price_ranges[x][0],
        price_ranges[x][1] + 1
    )
)

In [10]:
dim_product.head()

,product_id,product_name,aisle_id,department_id,aisle,department,unit_price
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks,112
1,2,All-Seasons Salt,104,13,spices seasonings,pantry,455
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages,290
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen,186
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry,91


In [11]:
dim_product.groupby("department")["unit_price"].agg(
    ["min", "max", "mean"]
).round(2)

,min,max,mean
department,,,
alcohol,203,2500,1369.96
babies,103,2999,1614.74
bakery,25,250,137.59
beverages,20,300,160.24
breakfast,30,400,207.05
bulk,559,4961,2890.34
canned goods,25,350,188.15
dairy eggs,30,250,141.00
deli,50,500,269.93


## Fact Order Product Design

Goal:
Create the transactional revenue table by combining purchased products with product pricing.

In [12]:
fact_order_product = (
    df_opp.merge(
        dim_product[
            [
                "product_id",
                "unit_price",
                "department",
                "aisle"
            ]
        ],
        on="product_id",
        how="left"
    )
)

In [13]:
fact_order_product["quantity"] = 1

fact_order_product["line_revenue"] = (
    fact_order_product["quantity"]
    * fact_order_product["unit_price"]
)

In [14]:
fact_order_product.shape

(32434489, 9)

In [15]:
fact_order_product.head()

,order_id,product_id,add_to_cart_order,reordered,unit_price,department,aisle,quantity,line_revenue
0,2,33120,1,1,41,dairy eggs,eggs,1,41
1,2,28985,2,1,92,produce,fresh vegetables,1,92
2,2,9327,3,0,338,pantry,spices seasonings,1,338
3,2,45918,4,1,94,pantry,oils vinegars,1,94
4,2,30035,5,0,413,pantry,baking ingredients,1,413


## Fact Orders Design

Goal:
Create an order-level fact table containing customer, timing, and revenue information.

In [16]:
order_metrics = (
    fact_order_product
    .groupby("order_id")
    .agg(
        total_items=("product_id", "count"),
        order_revenue=("line_revenue", "sum")
    )
    .reset_index()
)

In [17]:
fact_orders = (
    df_ord.merge(
        order_metrics,
        on="order_id",
        how="left"
    )
)

In [19]:
fact_orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,total_items,order_revenue
0,2539329,1,prior,1,2,8,NaN,5.0,1344.0
1,2398795,1,prior,2,3,7,15.0,6.0,764.0
2,473747,1,prior,3,3,12,21.0,5.0,881.0
3,2254736,1,prior,4,4,7,29.0,5.0,1385.0
4,431534,1,prior,5,4,15,28.0,8.0,803.0


In [20]:
fact_orders.shape

(3421083, 9)

## Customer Geography Design

Goal:
Assign each customer to a city to enable geographic revenue and customer analysis.

In [21]:
cities = [
    "Delhi",
    "Mumbai",
    "Bangalore",
    "Hyderabad",
    "Pune",
    "Chennai",
    "Kolkata",
    "Gurugram"
]

In [22]:
customers = pd.DataFrame(
    {"user_id": fact_orders["user_id"].unique()}
)

np.random.seed(42)

customers["city"] = np.random.choice(
    cities,
    size=len(customers)
)

In [24]:
customers.head()

,user_id,city
0,1,Kolkata
1,2,Hyderabad
2,3,Pune
3,4,Kolkata
4,5,Bangalore


In [25]:
customers["city"].value_counts()

city
Pune         26058
Kolkata      25901
Mumbai       25875
Hyderabad    25801
Chennai      25775
Delhi        25729
Gurugram     25613
Bangalore    25457
Name: count, dtype: int64

## Customer Segmentation Design

Customers are segmented based on their total order frequency to identify high-value and low-engagement customer groups.

Segmentation Rules:
- Occasional: 1–10 orders
- Regular: 11–30 orders
- VIP: More than 30 orders

In [33]:
customer_orders = (
    fact_orders
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "count")
    )
    .reset_index()
)

In [34]:
dim_customer = customers.merge(
    customer_orders,
    on="user_id",
    how="left"
)

In [35]:
dim_customer["customer_segment"] = pd.cut(
    dim_customer["total_orders"],
    bins=[0, 10, 30, float("inf")],
    labels=["Occasional", "Regular", "VIP"]
)

In [36]:
dim_customer.head()

,user_id,city,total_orders,customer_segment
0,1,Kolkata,11,Regular
1,2,Hyderabad,15,Regular
2,3,Pune,13,Regular
3,4,Kolkata,6,Occasional
4,5,Bangalore,5,Occasional


In [37]:
dim_customer["customer_segment"].value_counts()

customer_segment
Occasional    104513
Regular        72513
VIP            29183
Name: count, dtype: int64

## Hub Network Design

Create fulfillment hubs across cities to simulate quick-commerce order fulfillment operations.

Each city contains multiple fulfillment hubs responsible for processing and delivering customer orders.

In [38]:
hub_data = [
    ("DEL_01", "Delhi"),
    ("DEL_02", "Delhi"),
    ("DEL_03", "Delhi"),

    ("MUM_01", "Mumbai"),
    ("MUM_02", "Mumbai"),
    ("MUM_03", "Mumbai"),

    ("BLR_01", "Bangalore"),
    ("BLR_02", "Bangalore"),
    ("BLR_03", "Bangalore"),

    ("HYD_01", "Hyderabad"),
    ("HYD_02", "Hyderabad"),
    ("HYD_03", "Hyderabad"),

    ("PUN_01", "Pune"),
    ("PUN_02", "Pune"),
    ("PUN_03", "Pune"),

    ("CHE_01", "Chennai"),
    ("CHE_02", "Chennai"),
    ("CHE_03", "Chennai"),

    ("KOL_01", "Kolkata"),
    ("KOL_02", "Kolkata"),
    ("KOL_03", "Kolkata"),

    ("GUR_01", "Gurugram"),
    ("GUR_02", "Gurugram"),
    ("GUR_03", "Gurugram")
]

dim_hub = pd.DataFrame(
    hub_data,
    columns=["hub_id", "city"]
)

In [39]:
dim_hub

,hub_id,city
0,DEL_01,Delhi
1,DEL_02,Delhi
2,DEL_03,Delhi
3,MUM_01,Mumbai
4,MUM_02,Mumbai
5,MUM_03,Mumbai
6,BLR_01,Bangalore
7,BLR_02,Bangalore
8,BLR_03,Bangalore
9,HYD_01,Hyderabad


In [40]:
city_hubs = (
    dim_hub
    .groupby("city")["hub_id"]
    .apply(list)
    .to_dict()
)

np.random.seed(42)

dim_customer["hub_id"] = dim_customer["city"].apply(
    lambda x: np.random.choice(city_hubs[x])
)

In [41]:
dim_customer.head()

,user_id,city,total_orders,customer_segment,hub_id
0,1,Kolkata,11,Regular,KOL_03
1,2,Hyderabad,15,Regular,HYD_01
2,3,Pune,13,Regular,PUN_03
3,4,Kolkata,6,Occasional,KOL_03
4,5,Bangalore,5,Occasional,BLR_01


In [42]:
dim_customer["hub_id"].value_counts()

hub_id
PUN_01    8797
DEL_01    8750
MUM_02    8725
HYD_01    8689
KOL_02    8657
DEL_02    8648
CHE_01    8635
MUM_03    8632
PUN_03    8631
PUN_02    8630
KOL_03    8624
KOL_01    8620
HYD_02    8607
GUR_03    8599
CHE_02    8580
GUR_02    8570
CHE_03    8560
MUM_01    8518
BLR_03    8507
HYD_03    8505
BLR_01    8492
BLR_02    8458
GUR_01    8444
DEL_03    8331
Name: count, dtype: int64

## Order Enrichment

Enhance order-level data with customer geography and segmentation attributes to support city, hub, and customer-level analysis.

In [43]:
fact_orders = fact_orders.merge(
    dim_customer[
        [
            "user_id",
            "city",
            "hub_id",
            "customer_segment"
        ]
    ],
    on="user_id",
    how="left"
)

In [44]:
fact_orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,total_items,order_revenue,city,hub_id,customer_segment
0,2539329,1,prior,1,2,8,NaN,5.0,1344.0,Kolkata,KOL_03,Regular
1,2398795,1,prior,2,3,7,15.0,6.0,764.0,Kolkata,KOL_03,Regular
2,473747,1,prior,3,3,12,21.0,5.0,881.0,Kolkata,KOL_03,Regular
3,2254736,1,prior,4,4,7,29.0,5.0,1385.0,Kolkata,KOL_03,Regular
4,431534,1,prior,5,4,15,28.0,8.0,803.0,Kolkata,KOL_03,Regular


In [45]:
fact_orders.shape

(3421083, 12)

## Delivery Operations Simulation

Create delivery-level operational metrics to simulate quick-commerce fulfillment performance.

The delivery table supports operational KPI tracking, delay analysis, rider utilization, and hub performance monitoring.

Delivery times are generated using a weighted distribution to reflect realistic quick-commerce operations where most orders are delivered on time and only a small percentage experience significant delays.

In [54]:
delivery = fact_orders[
    [
        "order_id",
        "hub_id",
        "city"
    ]
].copy()

In [55]:
np.random.seed(42)

n = len(delivery)

delivery["delivery_time_minutes"] = np.concatenate([
    
    # 75% Delivered
    np.random.randint(8, 26, int(n * 0.75)),
    
    # 20% Delayed
    np.random.randint(26, 36, int(n * 0.20)),
    
    # 5% High Delay
    np.random.randint(36, 46, n - int(n * 0.75) - int(n * 0.20))
])

In [56]:
delivery = delivery.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [57]:
delivery["delivery_status"] = np.where(
    delivery["delivery_time_minutes"] <= 25,
    "Delivered",
    np.where(
        delivery["delivery_time_minutes"] <= 35,
        "Delayed",
        "High Delay"
    )
)

In [58]:
delivery["rider_id"] = np.random.randint(
    1,
    501,
    size=len(delivery)
)

delivery["rider_id"] = (
    "RID_"
    + delivery["rider_id"].astype(str)
)

In [59]:
fact_delivery = delivery

In [60]:
fact_delivery.head()

,order_id,hub_id,city,delivery_time_minutes,delivery_status,rider_id
0,1725437,BLR_01,Bangalore,16,Delivered,RID_182
1,1122088,KOL_02,Kolkata,28,Delayed,RID_399
2,352443,CHE_03,Chennai,21,Delivered,RID_90
3,2225015,PUN_01,Pune,26,Delayed,RID_364
4,1083635,BLR_01,Bangalore,22,Delivered,RID_375


In [62]:
fact_delivery["delivery_status"].value_counts(normalize=True) * 100

delivery_status
Delivered     74.999993
Delayed       19.999982
High Delay     5.000025
Name: proportion, dtype: float64

In [63]:
dim_product.to_csv("../data/processed/dim_product.csv", index=False)

dim_customer.to_csv("../data/processed/dim_customer.csv", index=False)

dim_hub.to_csv("../data/processed/dim_hub.csv", index=False)

fact_order_product.to_csv(
    "../data/processed/fact_order_product.csv",
    index=False
)

fact_orders.to_csv(
    "../data/processed/fact_orders.csv",
    index=False
)

fact_delivery.to_csv(
    "../data/processed/fact_delivery.csv",
    index=False
)

In [64]:
fact_order_product_sample = fact_order_product.sample(
    n=500000,
    random_state=42
)

fact_order_product_sample.shape

(500000, 9)

In [65]:
fact_order_product_sample.to_csv(
    "../data/processed/fact_order_product.csv",
    index=False
)

In [66]:
fact_orders_sample = fact_orders.sample(
    n=500000,
    random_state=42
)

fact_orders_sample.shape

(500000, 12)

In [67]:
fact_delivery_sample = fact_delivery[
    fact_delivery["order_id"].isin(
        fact_orders_sample["order_id"]
    )
]
fact_delivery_sample.shape

(500000, 6)

In [68]:
fact_orders_sample.to_csv(
    "../data/processed/fact_orders.csv",
    index=False
)

fact_delivery_sample.to_csv(
    "../data/processed/fact_delivery.csv",
    index=False
)